# YSSY Weather Data Pipeline

Sydney Airport (YSSY) area weather station data processing and wind forecast modelling.

## Structure

```
Part 0: Global Settings          <- imports, paths, parameters, run switches

Part 1: Data Processing
  1a  Merge stations
  1b  Crop to year range
  1c  Drop columns
  1d  Data quality analysis (missing timestamps, outages, availability)
  1e  Linear interpolation (single-step gaps)
  1f  Wind speed/direction -> U/V component conversion
  1g  Spline interpolation (larger gaps)
  1h  Interpolation verification

Part 2: Wind Forecast Model
  2a  Hyperparameter tuning (Optuna)
  2b  Train 96 individual LightGBM models
  2c  Evaluate all models
  2d  Plot forecast samples
```

---
# Part 0: Global Settings

All imports, paths, parameters, and run switches are defined here.  
Fill in paths before running. Set `RUN_XXX = True` for steps to execute.

## 0.1 Imports

In [ ]:
import pandas as pd
import numpy as np
import os
import glob
import random
import json
import time as _time
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from pathlib import Path
from datetime import timedelta
from scipy.interpolate import UnivariateSpline

# Data processing
try:
    matplotlib.use('Agg')
except ImportError:
    pass

# Model
try:
    import lightgbm as lgb
    from sklearn.multioutput import MultiOutputRegressor
    from sklearn.metrics import mean_squared_error, mean_absolute_error
    import optuna
    import joblib
    MODEL_DEPS_AVAILABLE = True
except ImportError:
    MODEL_DEPS_AVAILABLE = False
    print("Warning: lightgbm/optuna/joblib not installed. Model sections will not work.")

print("Imports complete.")

## 0.2 Paths

All folder/file paths. Leave empty strings for steps you don't need.

In [ ]:
# --- Data Processing Paths ---
MERGE_INPUT_FOLDER = ""              # e.g. "ProcessedData Manual"
MERGE_FILE1_NAME = ""                # e.g. "YSRI.txt"
MERGE_FILE2_NAME = ""                # e.g. "YSRI Old.txt"
MERGE_OUTPUT_FILE_NAME = ""          # e.g. "YSRI_merged.txt"

CROP_INPUT_FOLDER = ""               # e.g. "FinalData"
CROP_OUTPUT_FOLDER = ""              # e.g. "FinalData2000"

DROP_INPUT_FOLDER = ""               # e.g. "FinalData"
DROP_OUTPUT_FOLDER = ""              # e.g. "FinalData"

LINEAR_INTERP_INPUT_FOLDER = ""      # e.g. "ProcessedData"
LINEAR_INTERP_OUTPUT_FOLDER = ""     # e.g. "InterpolatedData0.5"

UV_INPUT_FOLDER = ""                 # e.g. "InterpolatedData0.5"
UV_OUTPUT_FOLDER = ""                # e.g. "UVComponentData"

SPLINE_INPUT_FOLDER = ""             # e.g. "UVComponentData"
SPLINE_OUTPUT_FOLDER = ""            # e.g. "SplineInterpolatedData"
SPLINE_PLOTS_FOLDER = ""             # e.g. "SplineInterpolationSamplePlots"

OUTAGE_INPUT_FOLDER = ""             # e.g. "InterpolatedData0.5"
OUTAGE_PLOTS_FOLDER = ""             # e.g. "OutageDistributionPlots_Grouped"

AVAILABILITY_INPUT_FOLDER = ""       # e.g. "ProcessedData"
AVAILABILITY_PLOTS_FOLDER = ""       # e.g. "StationElementPlots"

# --- Model Paths ---
MODEL_TRAIN_FILE = ""                # e.g. "data24/training_dataset.txt"
MODEL_VAL_FILE = ""                  # e.g. "data24/validation_dataset.txt"
MODEL_TEST_FILE = ""                 # e.g. "data24/test_dataset.txt"
YSSY_OBS_FILE = ""                   # e.g. "data/YSSY.txt"
MODEL_OUTPUT_DIR = ""                # e.g. "output_final_models"
MODEL_EVAL_OUTPUT_DIR = ""           # e.g. "output_model_evaluation"
MODEL_PLOTS_OUTPUT_DIR = ""          # e.g. "forecast_plots"

print("Paths configured.")

## 0.3 Data Processing Parameters

In [ ]:
# --- Time ---
TIMESTEP_MINUTES = 30
DATA_START_DATE = pd.Timestamp("2000-01-01 00:00:00")
DATA_END_DATE = pd.Timestamp("2024-12-31 23:30:00")

# --- Crop ---
CROP_START_YEAR = 2000
CROP_END_YEAR = 2000

# --- Drop Columns ---
COLUMNS_TO_DROP = ["max_gust_speed", "aws_flag", "wind_dir_recalc"]

# --- Weather Elements ---
WEATHER_PARAMS = [
    "air_temp", "dew_point", "wind_speed", "wind_dir",
    "max_gust_speed", "msl_pressure", "aws_flag"
]
ELEMENTS_TO_INTERPOLATE = [
    "air_temp", "dew_point", "wind_speed",
    "wind_dir", "max_gust_speed", "msl_pressure"
]
ELEMENTS_TO_SPLINE_INTERPOLATE = [
    "air_temp", "dew_point", "msl_pressure",
    "u_component", "v_component"
]
WEATHER_ELEMENTS_TO_ANALYZE = [
    "air_temp", "dew_point", "wind_speed", "wind_dir", "msl_pressure"
]

# --- Interpolation ---
START_YEAR_FILTER = None               # e.g. 2000 or None
MAX_GAP_STEPS_FOR_SPLINE = 5
SPLINE_CONTEXT_POINTS = 6
SPLINE_ORDER = 3
SPLINE_SMOOTHING_FACTOR = 1
NUM_SAMPLE_PLOTS = 50
PLOT_CONTEXT_HOURS = 12

# --- Outage Analysis ---
OUTAGE_START_YEAR = 2005

print("Data processing parameters loaded.")

## 0.4 Model Parameters

In [ ]:
# --- Model Architecture ---
TARGET_STATION_PREFIX = "YSSY"
NUM_FORECAST_STEPS = 48                # 48 steps = 24 hours
MODEL_TIMESTAMP_COL = "timestamp_t"
YSSY_OBS_TIMESTAMP_COL = "timestamp"

# --- Training ---
COMPONENT_TO_TRAIN = "both"             # "u", "v", or "both"
TRAIN_STEP_RANGE = range(1, 49)         # or subset e.g. range(1, 5)
EARLY_STOPPING_ROUNDS = 50
LGBM_FIT_VERBOSE_PERIOD = 250

# --- Tuning ---
TRAIN_SUBSAMPLE_FRACTION = 0.1
OPTUNA_N_TRIALS = 10

CHOSEN_HYPERPARAMETERS = {
    "learning_rate": 0.01,
    "num_leaves": 300,
    "max_depth": 40,
    "min_child_samples": 100,
    "subsample": 0.7,
    "colsample_bytree": 0.7,
    "reg_alpha": 0.5,
    "reg_lambda": 0.1,
    "n_estimators_max": 4000,
    "objective": "regression_l2",
    "metric": "l2",
    "random_state": 42,
    "n_jobs": -1,
    "verbose": -1,
}

# --- Plotting ---
NUM_RANDOM_PLOT_SAMPLES = 5
PLOT_SPECIFIC_DATE_RANGE = False
PLOT_START_DATETIME = "2024-01-01 00:00:00"
PLOT_END_DATETIME = "2024-01-31 23:00:00"

print("Model parameters loaded.")

## 0.5 Run Switches

In [ ]:
# --- Data Processing ---
RUN_MERGE = False
RUN_CROP = False
RUN_DROP = False
RUN_LINEAR_INTERP = False
RUN_UV_CONVERT = False
RUN_SPLINE_INTERP = False

# --- Model ---
RUN_TUNE = False
RUN_TRAIN = False
RUN_EVALUATE = False
RUN_PLOT = False

print("Run switches configured.")

---
# Part 1: Data Processing

Cleans and interpolates raw weather station data.  
Each sub-step reads `.txt` files, processes them, and saves output.

## 1a: Merge Station Data

Merges two station files (newer station data takes priority over older).

In [ ]:
def read_processed_file(filepath):
    try:
        df = pd.read_csv(filepath, na_values=['NaN'], parse_dates=['timestamp'],
                         infer_datetime_format=True)
        for col in WEATHER_PARAMS:
            if col in df.columns and df[col].dtype == 'object':
                df[col] = pd.to_numeric(df[col], errors='coerce')
        print(f"Read: {filepath}, shape: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"File not found: {filepath}")
        return None
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return None


def merge_weather_data(file1_path, file2_path, output_filepath):
    print(f"Merging:\n  File 1: {file1_path}\n  File 2: {file2_path}")
    df1 = read_processed_file(file1_path)
    df2 = read_processed_file(file2_path)
    if df1 is None or df2 is None:
        print("Aborting merge.")
        return None

    merged = pd.merge(df1, df2, on="timestamp", how="outer", suffixes=("_f1", "_f2"))
    print(f"Outer merge shape: {merged.shape}")

    for param in WEATHER_PARAMS:
        f1, f2 = f"{param}_f1", f"{param}_f2"
        if f2 in merged.columns and f1 in merged.columns:
            merged[param] = merged[f2].combine_first(merged[f1])
        elif f2 in merged.columns:
            merged[param] = merged[f2]
        elif f1 in merged.columns:
            merged[param] = merged[f1]
        else:
            merged[param] = pd.NA

    params_check = [p for p in WEATHER_PARAMS if p in merged.columns]
    merged['data_completeness'] = merged[params_check].notna().all(axis=1).astype(int)

    final_cols = ['timestamp'] + WEATHER_PARAMS + ['data_completeness']
    final_present = [c for c in final_cols if c in merged.columns]
    output_df = merged[final_present].sort_values(by="timestamp").reset_index(drop=True)

    out_dir = os.path.dirname(output_filepath)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    output_df.to_csv(output_filepath, index=False, na_rep='NaN')
    print(f"Saved: {output_filepath}, shape: {output_df.shape}")
    return output_df


if RUN_MERGE:
    assert MERGE_INPUT_FOLDER and MERGE_FILE1_NAME and MERGE_FILE2_NAME and MERGE_OUTPUT_FILE_NAME
    p1 = os.path.join(MERGE_INPUT_FOLDER, MERGE_FILE1_NAME)
    p2 = os.path.join(MERGE_INPUT_FOLDER, MERGE_FILE2_NAME)
    po = os.path.join(MERGE_INPUT_FOLDER, MERGE_OUTPUT_FILE_NAME)
    if os.path.exists(p1) and os.path.exists(p2):
        merged = merge_weather_data(p1, p2, po)
        if merged is not None:
            display(merged.head())
    else:
        print(f"Files not found.\n  {p1}\n  {p2}")
else:
    print("RUN_MERGE is False. Skipping.")

## 1b: Crop to Year Range

In [ ]:
def filter_files_by_year_range(input_folder, output_folder, start_year, end_year):
    if not os.path.isdir(input_folder):
        print(f"Input folder not found: {input_folder}")
        return
    os.makedirs(output_folder, exist_ok=True)
    start_dt = pd.Timestamp(f"{start_year}-01-01")
    end_dt = pd.Timestamp(f"{end_year}-12-31 23:59:59")
    print(f"Filtering {start_year}-{end_year}...")

    for fp in glob.glob(os.path.join(input_folder, "*.txt")):
        fname = os.path.basename(fp)
        out = os.path.join(output_folder, fname)
        print(f"\n  {fname}")
        try:
            df = pd.read_csv(fp)
            ts_col = 'timestamp' if 'timestamp' in df.columns else df.columns[0]
            df[ts_col] = pd.to_datetime(df[ts_col], errors='coerce')
            orig = len(df)
            df.dropna(subset=[ts_col], inplace=True)
            filtered = df[(df[ts_col] >= start_dt) & (df[ts_col] <= end_dt)]
            filtered.to_csv(out, index=False)
            print(f"    {orig} -> {len(filtered)} rows. Saved: {out}")
        except Exception as e:
            print(f"    Error: {e}")
    print("\nYear filtering complete.")


if RUN_CROP:
    assert CROP_INPUT_FOLDER and CROP_OUTPUT_FOLDER
    filter_files_by_year_range(CROP_INPUT_FOLDER, CROP_OUTPUT_FOLDER, CROP_START_YEAR, CROP_END_YEAR)
else:
    print("RUN_CROP is False. Skipping.")

## 1c: Drop Unwanted Columns

In [ ]:
def drop_columns_from_files(input_folder, output_folder, cols_to_drop):
    if not os.path.isdir(input_folder):
        print(f"Input folder not found: {input_folder}")
        return
    os.makedirs(output_folder, exist_ok=True)
    print(f"Dropping: {cols_to_drop}")

    for fp in glob.glob(os.path.join(input_folder, "*.txt")):
        fname = os.path.basename(fp)
        out = os.path.join(output_folder, fname)
        try:
            df = pd.read_csv(fp)
            existing = [c for c in cols_to_drop if c in df.columns]
            if existing:
                print(f"  {fname}: dropping {existing}")
            df.drop(columns=existing, errors='ignore').to_csv(out, index=False)
        except Exception as e:
            print(f"  {fname}: Error: {e}")
    print("Column dropping complete.")


if RUN_DROP:
    assert DROP_INPUT_FOLDER and DROP_OUTPUT_FOLDER
    drop_columns_from_files(DROP_INPUT_FOLDER, DROP_OUTPUT_FOLDER, COLUMNS_TO_DROP)
else:
    print("RUN_DROP is False. Skipping.")

## 1d: Data Quality Analysis

Diagnostic tools for data completeness. These cells can be run independently.

In [ ]:
# --- Missing Timestamps ---
def find_missing_timestamps(data_folder, file1_name, file2_name=None,
                            start_date=None, end_date=None, timestep_min=30):
    if start_date is None:
        start_date = DATA_START_DATE
    if end_date is None:
        end_date = DATA_END_DATE

    expected = set(pd.date_range(start=start_date, end=end_date, freq=f"{timestep_min}T"))
    print(f"Expected timestamps: {len(expected)}")

    df1 = pd.read_csv(os.path.join(data_folder, file1_name))
    df1['timestamp'] = pd.to_datetime(df1['timestamp'])
    df1 = df1.set_index('timestamp')
    ts1 = set(df1.index)
    print(f"{file1_name}: {len(ts1)} timestamps")

    missing = sorted(expected - ts1)
    print(f"Missing in {file1_name}: {len(missing)}")
    if missing:
        print("  First 10:")
        for ts in missing[:10]:
            print(f"    {ts}")

    if file2_name:
        df2 = pd.read_csv(os.path.join(data_folder, file2_name))
        df2['timestamp'] = pd.to_datetime(df2['timestamp'])
        df2 = df2.set_index('timestamp')
        ts2 = set(df2.index)
        print(f"\n{file2_name}: {len(ts2)} timestamps")
        print(f"In {file2_name} but not {file1_name}: {len(ts2 - ts1)}")
        print(f"In {file1_name} but not {file2_name}: {len(ts1 - ts2)}")

# Uncomment to run:
# find_missing_timestamps("", "YSNW.txt", "YSCN.txt")
print("Missing timestamps function defined. Uncomment above to run.")

In [ ]:
# --- Outage Length Distribution ---
OUTAGE_BINS = [
    (1, 2, "30 min"), (2, 3, "1 hr"), (3, 5, "1.5-2 hr"),
    (5, 13, "2-6 hr"), (13, 49, "6-24 hr"), (49, 337, "1-7 day"),
    (337, 1441, "7d-1mo"), (1441, 17521, "1mo-1yr"), (17521, float('inf'), ">1 yr")
]
OUTAGE_LABELS = [b[2] for b in OUTAGE_BINS]


def find_outage_durations(series):
    outages, cur = [], 0
    for v in series.isna():
        if v:
            cur += 1
        else:
            if cur > 0:
                outages.append(cur)
            cur = 0
    if cur > 0:
        outages.append(cur)
    return outages


def plot_outage_summary(folder_path, output_folder, start_year=None):
    os.makedirs(output_folder, exist_ok=True)
    elements = WEATHER_ELEMENTS_TO_ANALYZE
    n_el = len(elements)
    n_bins = len(OUTAGE_LABELS)
    bw = 0.8 / n_el
    colors = plt.cm.get_cmap('viridis', n_el)

    for fp in glob.glob(os.path.join(folder_path, "*.txt")):
        sid = os.path.basename(fp).split('.')[0]
        print(f"\n--- {sid} ---")
        df = pd.read_csv(fp, parse_dates=['timestamp'], na_values=['NaN'])
        if df.empty:
            continue
        if start_year:
            df = df[df['timestamp'].dt.year >= start_year]
        if df.empty:
            continue
        total = len(df)

        binned = pd.DataFrame(index=OUTAGE_LABELS)
        uptime = {}
        for el in elements:
            if el not in df.columns:
                binned[el] = 0; uptime[el] = "N/A"; continue
            uptime[el] = f"{df[el].notna().sum() / total * 100:.1f}%"
            lengths = find_outage_durations(df[el])
            counts = {b[2]: 0 for b in OUTAGE_BINS}
            for d in lengths:
                for lo, hi, lbl in OUTAGE_BINS:
                    if lo <= d < hi:
                        counts[lbl] += 1; break
            binned[el] = pd.Series(counts, index=OUTAGE_LABELS)

        fig, ax = plt.subplots(figsize=(16, 8))
        x = np.arange(n_bins)
        max_y = max(10, binned.iloc[1:].max().max() * 1.25) if len(binned) > 1 else 10
        ax.set_ylim(0, max_y)

        for i, el in enumerate(elements):
            if el not in binned.columns:
                continue
            pos = x + bw * (i - n_el / 2 + 0.5)
            bars = ax.bar(pos, binned[el].values, bw,
                          label=el.replace("_", " ").title(), color=colors(i / n_el))
            for bi, bar in enumerate(bars):
                yv = bar.get_height()
                if yv > 0:
                    ax.text(bar.get_x() + bar.get_width() / 2, yv, int(yv),
                            ha='center', va='bottom', fontsize=7)

        title = f'Outage Distribution - {sid}'
        if start_year:
            title += f' (since {start_year})'
        ax.set_title(title, fontsize=16, pad=20)
        ax.set_xlabel('Duration'); ax.set_ylabel('Count')
        ax.set_xticks(x); ax.set_xticklabels(OUTAGE_LABELS, rotation=45, ha="right")
        ax.grid(axis='y', linestyle='--', alpha=0.6)
        ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=9)

        ut = "Uptime:\n" + "\n".join(f"{e}: {uptime.get(e, 'N/A')}" for e in elements)
        fig.text(0.01, 0.98, ut, transform=fig.transFigure, fontsize=8, va='top',
                 bbox=dict(boxstyle='round,pad=0.5', fc='lightyellow', alpha=0.7))
        plt.tight_layout(rect=[0.05, 0, 0.85, 0.95])
        plt.savefig(os.path.join(output_folder, f"{sid}_outages.png"))
        plt.show(); plt.close(fig)

# Uncomment to run:
# if OUTAGE_INPUT_FOLDER and OUTAGE_PLOTS_FOLDER:
#     plot_outage_summary(OUTAGE_INPUT_FOLDER, OUTAGE_PLOTS_FOLDER, start_year=OUTAGE_START_YEAR)
print("Outage function defined. Uncomment above to run.")

In [ ]:
# --- Monthly Availability ---
AVAILABILITY_ELEMENTS = ["air_temp", "dew_point", "wind_speed", "wind_dir", "msl_pressure"]

def plot_availability(folder_path, output_folder="StationPlots"):
    os.makedirs(output_folder, exist_ok=True)
    for fp in glob.glob(os.path.join(folder_path, "*.txt")):
        sid = os.path.basename(fp).split('.')[0]
        df = pd.read_csv(fp, parse_dates=['timestamp'], na_values=['NaN'])
        if 'timestamp' not in df.columns or df.empty:
            continue
        for el in AVAILABILITY_ELEMENTS:
            if el not in df.columns:
                continue
        df_h = df[df['timestamp'].dt.minute == 0].copy()
        if df_h.empty:
            continue
        df_h['ym'] = df_h['timestamp'].dt.to_period('M')

        plt.figure(figsize=(15, 8))
        has_data = False
        for el in AVAILABILITY_ELEMENTS:
            if el not in df_h.columns:
                continue
            m = df_h.groupby('ym').agg(total=('timestamp', 'count'),
                                       avail=(el, lambda x: x.notna().sum())).reset_index()
            if m.empty:
                continue
            m['pct'] = 0.0
            mask = m['total'] > 0
            m.loc[mask, 'pct'] = m.loc[mask, 'avail'] / m.loc[mask, 'total'] * 100
            m['dt'] = m['ym'].dt.to_timestamp()
            plt.plot(m['dt'], m['pct'], label=el.replace('_', ' ').title(), marker='.', linestyle='-')
            has_data = True

        if not has_data:
            plt.close(); continue
        plt.title(f'Monthly Availability - {sid}'); plt.ylim(0, 105)
        plt.xlabel('Month'); plt.ylabel('Available (%)')
        plt.grid(True, linestyle='--', alpha=0.5); plt.legend(title='Element')
        ax = plt.gca()
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.xaxis.set_major_locator(mdates.AutoDateLocator(minticks=8, maxticks=20))
        plt.xticks(rotation=45, ha='right'); plt.tight_layout()
        plt.savefig(os.path.join(output_folder, f"{sid}_availability.png"))
        plt.show(); plt.close()

# Uncomment to run:
# if AVAILABILITY_INPUT_FOLDER and AVAILABILITY_PLOTS_FOLDER:
#     plot_availability(AVAILABILITY_INPUT_FOLDER, AVAILABILITY_PLOTS_FOLDER)
print("Availability function defined. Uncomment above to run.")

## 1e: Linear Interpolation (Single-Step Gaps)

Fills gaps of exactly 1 missing timestep: `(prev + next) / 2`.

In [ ]:
def interpolate_single_step_gaps(series):
    result = series.copy()
    is_na = series.isna()
    n, count = len(series), 0
    for i in range(1, n - 1):
        if not is_na.iloc[i - 1] and is_na.iloc[i] and not is_na.iloc[i + 1]:
            result.iloc[i] = (series.iloc[i - 1] + series.iloc[i + 1]) / 2.0
            count += 1
    if count > 0:
        print(f"    Interpolated {count} gaps for '{series.name}'")
    return result


def run_linear_interpolation(input_folder, output_folder, start_year=None):
    os.makedirs(output_folder, exist_ok=True)
    for fp in glob.glob(os.path.join(input_folder, "*.txt")):
        fname = os.path.basename(fp)
        sid = fname.split('.')[0]
        print(f"\n--- {sid} ---")
        try:
            df = pd.read_csv(fp, parse_dates=['timestamp'], na_values=['NaN'])
            if df.empty:
                continue
            if start_year:
                df = df[df['timestamp'].dt.year >= start_year]
                if df.empty:
                    continue
            for el in ELEMENTS_TO_INTERPOLATE:
                if el in df.columns:
                    if not pd.api.types.is_numeric_dtype(df[el]):
                        df[el] = pd.to_numeric(df[el], errors='coerce')
                    df[el] = interpolate_single_step_gaps(df[el])
            if 'data_completeness' in df.columns:
                cols = [e for e in ELEMENTS_TO_INTERPOLATE if e in df.columns]
                if cols:
                    df['data_completeness'] = (df[cols].notna().all(axis=1) & df['timestamp'].notna()).astype(int)
            df.to_csv(os.path.join(output_folder, fname), index=False, na_rep='NaN', float_format='%.1f')
            print(f"  Saved: {fname}")
        except Exception as e:
            print(f"  Error: {e}")
    print("\nLinear interpolation complete.")


if RUN_LINEAR_INTERP:
    assert LINEAR_INTERP_INPUT_FOLDER and LINEAR_INTERP_OUTPUT_FOLDER
    run_linear_interpolation(LINEAR_INTERP_INPUT_FOLDER, LINEAR_INTERP_OUTPUT_FOLDER, START_YEAR_FILTER)
else:
    print("RUN_LINEAR_INTERP is False. Skipping.")

## 1f: Wind Speed/Direction -> U/V Components

Meteorological convention: `u = -speed * sin(dir)`, `v = speed * cos(dir)`.  
Original wind_speed and wind_dir columns are replaced.

In [ ]:
def convert_to_uv(input_folder, output_folder, start_year=None):
    os.makedirs(output_folder, exist_ok=True)
    for fp in glob.glob(os.path.join(input_folder, "*.txt")):
        fname = os.path.basename(fp)
        sid = fname.split('.')[0]
        print(f"\n--- {sid} ---")
        try:
            df = pd.read_csv(fp, parse_dates=['timestamp'], na_values=['NaN'])
            if df.empty:
                continue
            if start_year:
                df = df[df['timestamp'].dt.year >= start_year]
                if df.empty:
                    continue
            if 'wind_speed' not in df.columns or 'wind_dir' not in df.columns:
                print(f"  Wind columns missing. Saving as is.")
                df.to_csv(os.path.join(output_folder, fname), index=False, na_rep='NaN', float_format='%.2f')
                continue

            df['wind_speed'] = pd.to_numeric(df['wind_speed'], errors='coerce')
            df['wind_dir'] = pd.to_numeric(df['wind_dir'], errors='coerce')
            dir_rad = np.deg2rad(df['wind_dir'].astype(float))
            df['u_component'] = -df['wind_speed'] * np.sin(dir_rad)
            df['v_component'] = df['wind_speed'] * np.cos(dir_rad)
            nan_mask = df['wind_speed'].isna() | df['wind_dir'].isna()
            df.loc[nan_mask, 'u_component'] = np.nan
            df.loc[nan_mask, 'v_component'] = np.nan

            keep = [c for c in df.columns if c not in ['wind_speed', 'wind_dir']]
            df[keep].to_csv(os.path.join(output_folder, fname), index=False, na_rep='NaN', float_format='%.2f')
            print(f"  U/V saved.")
        except Exception as e:
            print(f"  Error: {e}")
    print("\nUV conversion complete.")


if RUN_UV_CONVERT:
    assert UV_INPUT_FOLDER and UV_OUTPUT_FOLDER
    convert_to_uv(UV_INPUT_FOLDER, UV_OUTPUT_FOLDER, START_YEAR_FILTER)
else:
    print("RUN_UV_CONVERT is False. Skipping.")

## 1g: Spline Interpolation

Fills NaN gaps of up to `MAX_GAP_STEPS_FOR_SPLINE` steps using cubic spline.  
Interpolates U/V components, then recalculates wind speed/direction.  
Generates sample plots showing interpolation quality.

In [ ]:
def spline_interp_gap(series, gap_start, gap_end, ctx, k=3, s=0):
    n = len(series)
    x_known, y_known = [], []
    for (s_start, s_end) in [(max(0, gap_start - ctx), gap_start - 1),
                              (gap_end + 1, min(n - 1, gap_end + ctx))]:
        if s_end >= s_start:
            chunk = series.iloc[s_start:s_end + 1].dropna()
            if not chunk.empty:
                x_known.extend([series.index.get_loc(i) for i in chunk.index])
                y_known.extend(chunk.values.tolist())
    if len(x_known) < k + 1:
        return None
    x_arr = np.array(x_known); y_arr = np.array(y_known)
    order = np.argsort(x_arr)
    ux, idx = np.unique(x_arr[order], return_index=True)
    if len(ux) < k + 1:
        return None
    try:
        sp = UnivariateSpline(ux, y_arr[order][idx], k=k, s=s)
        ilocs = np.arange(gap_start, gap_end + 1)
        return pd.Series(sp(ilocs), index=series.index[ilocs])
    except Exception:
        return None


def apply_spline_to_series(series_orig, name=""):
    series = series_orig.copy()
    is_na = series_orig.isna()
    events = []
    gap_start = -1
    for i in range(len(series)):
        if is_na.iloc[i] and gap_start == -1:
            gap_start = i
        elif (not is_na.iloc[i] or i == len(series) - 1) and gap_start != -1:
            gap_end = (i - 1) if not is_na.iloc[i] else i
            gap_len = gap_end - gap_start + 1
            if 0 < gap_len <= MAX_GAP_STEPS_FOR_SPLINE:
                vals = spline_interp_gap(series_orig, gap_start, gap_end,
                                          SPLINE_CONTEXT_POINTS, k=SPLINE_ORDER, s=SPLINE_SMOOTHING_FACTOR)
                if vals is not None and not vals.empty:
                    series.loc[vals.index] = vals.values
                    events.append({'time_start_gap': series_orig.index[gap_start],
                                   'time_end_gap': series_orig.index[gap_end],
                                   'method': f'spline(k={SPLINE_ORDER},s={SPLINE_SMOOTHING_FACTOR})',
                                   'len_steps': gap_len})
            gap_start = -1
    return series, events


def run_spline_pipeline(input_folder, output_folder, plot_folder, start_year=None):
    os.makedirs(output_folder, exist_ok=True)
    os.makedirs(plot_folder, exist_ok=True)
    all_events = []

    for fp in glob.glob(os.path.join(input_folder, "*.txt")):
        fname = os.path.basename(fp)
        sid = fname.split('.')[0]
        print(f"\n--- {sid} ---")
        try:
            df = pd.read_csv(fp, parse_dates=['timestamp'], na_values=['NaN'])
            if df.empty:
                continue
            if start_year:
                df = df[df['timestamp'].dt.year >= start_year]
                if df.empty:
                    continue
            df = df.set_index('timestamp')
            df_save = df.copy()
            stn_events = []

            for el in ELEMENTS_TO_SPLINE_INTERPOLATE:
                if el not in df.columns:
                    continue
                print(f"  Spline: {el}")
                if not pd.api.types.is_numeric_dtype(df[el]):
                    df[el] = pd.to_numeric(df[el], errors='coerce')
                interp, evts = apply_spline_to_series(df[el], name=el)
                df_save[el] = interp
                for e in evts:
                    e['station_id'] = sid; e['element'] = el
                    stn_events.append(e)

            if 'u_component' in df_save.columns and 'v_component' in df_save.columns:
                print(f"  Recalculating wind from U/V...")
                df_save['wind_speed_recalc'] = np.sqrt(df_save['u_component']**2 + df_save['v_component']**2)
                dir_rad = np.arctan2(-df_save['u_component'], -df_save['v_component'])
                df_save['wind_dir_recalc'] = (np.rad2deg(dir_rad) + 360) % 360
                df_save['wind_dir_recalc'] = df_save['wind_dir_recalc'].where(
                    df_save['wind_speed_recalc'].notna() & (df_save['wind_speed_recalc'] > 0.01), np.nan)

            if 'data_completeness' in df_save.columns:
                chk = [e for e in ["air_temp", "dew_point", "msl_pressure",
                                    "wind_speed_recalc", "wind_dir_recalc"] if e in df_save.columns]
                if chk:
                    df_save['data_completeness'] = df_save[chk].notna().all(axis=1).astype(int)

            df_save.reset_index().to_csv(os.path.join(output_folder, fname), index=False, na_rep='NaN', float_format='%.3f')
            print(f"  Saved: {fname}")
            all_events.extend(stn_events)
        except Exception as e:
            print(f"  Error: {e}")
            import traceback; traceback.print_exc()

    print(f"\nSpline interpolation complete. {len(all_events)} events recorded.")


if RUN_SPLINE_INTERP:
    assert SPLINE_INPUT_FOLDER and SPLINE_OUTPUT_FOLDER and SPLINE_PLOTS_FOLDER
    run_spline_pipeline(SPLINE_INPUT_FOLDER, SPLINE_OUTPUT_FOLDER, SPLINE_PLOTS_FOLDER, START_YEAR_FILTER)
else:
    print("RUN_SPLINE_INTERP is False. Skipping.")

## 1h: Interpolation Verification

Tests the spline interpolation approach on synthetic data and reports RMSE.

In [ ]:
np.random.seed(42)
n_pts = 48
t = np.arange(n_pts)
data_orig = np.sin(t / ((n_pts - 1) / (2 * np.pi))) + t / 20 + np.random.normal(0, 0.3, n_pts)

gap_start, n_miss = 20, 10
data_miss = data_orig.copy()
data_miss[gap_start:gap_start + n_miss] = np.nan

s_miss = pd.Series(data_miss, index=t)
s_interp = s_miss.copy()
gap_idx = s_interp.index[gap_start:gap_start + n_miss]

pts_before = s_miss.iloc[:gap_start].dropna().tail(SPLINE_CONTEXT_POINTS)
pts_after = s_miss.iloc[gap_start + n_miss:].dropna().head(SPLINE_CONTEXT_POINTS)
local = pd.concat([pts_before, pts_after])

if len(local) >= SPLINE_ORDER + 1:
    sp = UnivariateSpline(local.index.values, local.values, s=SPLINE_SMOOTHING_FACTOR, k=SPLINE_ORDER)
    s_interp.loc[gap_idx] = sp(gap_idx.values)
else:
    print(f"Not enough points ({len(local)}). Fallback to linear.")
    tmp = pd.concat([pts_before, pd.Series(index=gap_idx, dtype=float), pts_after])
    s_interp.loc[gap_idx] = tmp.interpolate(method='linear').loc[gap_idx]

plt.figure(figsize=(15, 8))
plt.plot(t, data_orig, 'ko-', alpha=0.3, markersize=3, label='Original')
plt.plot(t, data_miss, 'bo', markersize=6, label='With Gap')
if len(local) >= SPLINE_ORDER + 1:
    plt.plot(local.index, local.values, 'ms', markersize=8, markerfacecolor='none', label=f'Context ({SPLINE_CONTEXT_POINTS}x2)')
plt.plot(t, s_interp.values, 'gD--', linewidth=2, markersize=4, label=f'Spline (k={SPLINE_ORDER}, s={SPLINE_SMOOTHING_FACTOR})')
plt.axvline(gap_start, color='gray', linestyle='--', alpha=0.7)
plt.axvline(gap_start + n_miss - 1, color='gray', linestyle='--', alpha=0.7, label='Gap bounds')
plt.title(f'Spline Interpolation Test (gap={n_miss} steps, ctx={SPLINE_CONTEXT_POINTS})')
plt.xlabel('Step'); plt.ylabel('Value'); plt.legend(); plt.grid(True); plt.tight_layout()
plt.show()

rmse = np.sqrt(np.mean((s_interp.loc[gap_idx].values - data_orig[gap_start:gap_start + n_miss]) ** 2))
print(f"RMSE: {rmse:.4f}")

---
# Part 2: Wind Forecast Model

Uses **LightGBM** to forecast YSSY wind U/V components up to 24 hours ahead.  
**96 independent models** (48 time steps x 2 components), each with early stopping.

```
2a Tune (Optuna) -> 2b Train (96 models) -> 2c Evaluate (MAE/MSE) -> 2d Plot
```

## 2a: Hyperparameter Tuning (Optuna)

Tunes hyperparameters for a **single target column**.  
Change `TUNE_TARGET_COL` to tune different forecast steps. Run multiple times as needed.

In [ ]:
def _get_target_cols(file_path, ts_col, prefix):
    cols = pd.read_csv(file_path, nrows=0).columns
    return sorted([c for c in cols if c.startswith(f"{prefix}_u_forecast_t_plus_")
                    or c.startswith(f"{prefix}_v_forecast_t_plus_")],
                   key=lambda x: (int(x.split('_t_plus_')[-1]), x.split('_')[1]))

def _get_feature_cols(file_path, ts_col, prefix):
    cols = pd.read_csv(file_path, nrows=0).columns
    targets = _get_target_cols(file_path, ts_col, prefix)
    return [c for c in cols if c != ts_col and c not in targets]

def load_single_target(train_path, val_path, target_col, ts_col, prefix, subsample=1.0):
    feat = _get_feature_cols(train_path, ts_col, prefix)
    df_tr = pd.read_csv(train_path)
    X_tr, y_tr = df_tr[feat], df_tr[target_col]
    if 0 < subsample < 1.0:
        X_tr = X_tr.sample(frac=subsample, random_state=42)
        y_tr = y_tr.loc[X_tr.index]
    df_va = pd.read_csv(val_path)
    X_va, y_va = df_va[feat], df_va[target_col]
    print(f"  {target_col}: train {X_tr.shape}, val {X_va.shape}")
    return X_tr, y_tr, X_va, y_va

def run_tuning(target_col, train_path, val_path, ts_col, prefix,
               subsample=0.1, n_trials=10):
    X_tr, y_tr, X_va, y_va = load_single_target(
        train_path, val_path, target_col, ts_col, prefix, subsample)

    def objective(trial):
        params = {
            'objective': 'regression_l2', 'metric': 'l2',
            'n_estimators': trial.suggest_int('n_estimators', 500, 2000, step=100),
            'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 20, 500),
            'max_depth': trial.suggest_int('max_depth', 5, 30),
            'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.4, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'random_state': 42, 'n_jobs': -1,
        }
        m = lgb.LGBMRegressor(**params)
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric='l2',
              callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                         lgb.log_evaluation(50)])
        score = mean_squared_error(y_va, m.predict(X_va))
        trial.set_user_attr("best_iter", m.best_iteration_)
        return score

    study = optuna.create_study(direction="minimize")
    t0 = _time.time()
    study.optimize(objective, n_trials=n_trials)
    dur = _time.time() - t0

    best = study.best_params
    best["best_iteration_"] = study.best_trial.user_attrs["best_iter"]
    best["val_mse"] = study.best_value
    trials_df = study.trials_dataframe().sort_values("value")

    print(f"\nTuning done in {dur:.1f}s. Best MSE: {study.best_value:.6f}, iter: {best['best_iteration_']}")
    return best, trials_df


if RUN_TUNE:
    assert MODEL_TRAIN_FILE and MODEL_VAL_FILE
    TUNE_TARGET_COL = "YSSY_u_forecast_t_plus_1"  # Change as needed
    best, trials = run_tuning(TUNE_TARGET_COL, MODEL_TRAIN_FILE, MODEL_VAL_FILE,
                               MODEL_TIMESTAMP_COL, TARGET_STATION_PREFIX,
                               TRAIN_SUBSAMPLE_FRACTION, OPTUNA_N_TRIALS)
    for k, v in best.items():
        print(f"  {k}: {v}")
    display(trials[['number', 'value', 'params_learning_rate', 'params_num_leaves',
                    'user_attrs_best_iter']].head())
    if MODEL_EVAL_OUTPUT_DIR:
        d = Path(MODEL_EVAL_OUTPUT_DIR); d.mkdir(parents=True, exist_ok=True)
        safe = TUNE_TARGET_COL.replace("+", "p")
        json.dump(best, open(d / f"best_params_{safe}.json", 'w'), indent=4)
        trials.to_csv(d / f"all_trials_{safe}.csv", index=False)
else:
    print("RUN_TUNE is False. Skipping.")

## 2b: Train Final Individual Models

Trains independent LightGBM models for each target. Saved in `u_models/` and `v_models/` subdirectories.

In [ ]:
def train_all(component, train_path, val_path, ts_col, prefix, hps,
              output_dir, step_range=None):
    if step_range is None:
        step_range = range(1, 49)
    comp_dir = Path(output_dir) / f"{component}_models"
    comp_dir.mkdir(parents=True, exist_ok=True)
    print(f"\n=== Training {component.upper()} models -> {comp_dir} ===")

    feat = _get_feature_cols(train_path, ts_col, prefix)
    df_tr = pd.read_csv(train_path)
    df_va = pd.read_csv(val_path)
    X_va = df_va[feat]

    total_t, results = 0, []
    for step in step_range:
        tgt = f"{prefix}_{component}_forecast_t_plus_{step}"
        if tgt not in df_tr.columns or tgt not in df_va.columns:
            print(f"  Skip {tgt}: not in data")
            continue

        X_tr = df_tr[feat]
        y_tr = df_tr[tgt]
        y_va = df_va[tgt]

        params = hps.copy()
        params['n_estimators'] = params.pop('n_estimators_max')
        m = lgb.LGBMRegressor(**params)
        t0 = _time.time()
        m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], eval_metric=hps.get('metric', 'l2'),
              callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False),
                         lgb.log_evaluation(LGBM_FIT_VERBOSE_PERIOD)])
        elapsed = _time.time() - t0
        total_t += elapsed

        path = comp_dir / f"{tgt}.joblib"
        joblib.dump(m, path)
        results.append({'step': step, 'component': component,
                         'best_iter': m.best_iteration_, 'time_s': round(elapsed, 1)})
        print(f"  {tgt}: {elapsed:.1f}s, iter={m.best_iteration_}")

    print(f"\n=== {component.upper()}: {len(results)} models in {total_t:.1f}s ({total_t/60:.1f}min) ===")
    return pd.DataFrame(results)


if RUN_TRAIN:
    assert MODEL_TRAIN_FILE and MODEL_VAL_FILE and MODEL_OUTPUT_DIR
    comps = ["u", "v"] if COMPONENT_TO_TRAIN == "both" else [COMPONENT_TO_TRAIN]
    all_res = []
    for comp in comps:
        df_res = train_all(comp, MODEL_TRAIN_FILE, MODEL_VAL_FILE,
                           MODEL_TIMESTAMP_COL, TARGET_STATION_PREFIX,
                           CHOSEN_HYPERPARAMETERS, MODEL_OUTPUT_DIR, TRAIN_STEP_RANGE)
        all_res.append(df_res)
    if all_res:
        display(pd.concat(all_res, ignore_index=True))
else:
    print("RUN_TRAIN is False. Skipping.")

## 2c: Evaluate All Models

Loads all models, predicts on test set, computes per-step MAE/MSE.  
Generates dual-panel plot of error metrics vs. forecast lead time.

In [ ]:
def load_all_models(models_dir):
    loaded, feat_names = {}, None
    for comp in ['u', 'v']:
        d = Path(models_dir) / f"{comp}_models"
        if not d.exists():
            continue
        for mf in sorted(d.glob("*.joblib")):
            try:
                m = joblib.load(mf)
                loaded[mf.stem] = m
                if feat_names is None:
                    if hasattr(m, 'feature_name_'):
                        feat_names = m.feature_name_
                    elif hasattr(m, 'booster_') and hasattr(m.booster_, 'feature_name'):
                        feat_names = m.booster_.feature_name()
            except Exception as e:
                print(f"  Error loading {mf}: {e}")
    print(f"Loaded {len(loaded)} models.")
    return loaded, feat_names


def predict_all(loaded, X_test, num_steps, prefix):
    preds = {}
    n = len(X_test)
    for s in range(1, num_steps + 1):
        for comp in ['u', 'v']:
            nm = f"{prefix}_{comp}_forecast_t_plus_{s}"
            preds[nm] = loaded[nm].predict(X_test) if nm in loaded else np.full(n, np.nan)
    return pd.DataFrame(preds)


if RUN_EVALUATE:
    assert MODEL_OUTPUT_DIR and MODEL_TEST_FILE
    loaded, feat_names = load_all_models(MODEL_OUTPUT_DIR)
    if not loaded:
        raise RuntimeError("No models loaded.")

    test_df = pd.read_csv(MODEL_TEST_FILE)
    print(f"Test shape: {test_df.shape}")

    if feat_names:
        X_test = test_df[feat_names]
    else:
        tgts = _get_target_cols(MODEL_TEST_FILE, MODEL_TIMESTAMP_COL, TARGET_STATION_PREFIX)
        X_test = test_df[[c for c in test_df.columns if c != MODEL_TIMESTAMP_COL and c not in tgts]]

    y_cols = []
    for s in range(1, NUM_FORECAST_STEPS + 1):
        y_cols += [f"{TARGET_STATION_PREFIX}_u_forecast_t_plus_{s}",
                   f"{TARGET_STATION_PREFIX}_v_forecast_t_plus_{s}"]
    y_test = test_df[y_cols]
    preds = predict_all(loaded, X_test, NUM_FORECAST_STEPS, TARGET_STATION_PREFIX)[y_test.columns]

    rows, mae_s, mse_s, cnt = [], 0, 0, 0
    for s in range(1, NUM_FORECAST_STEPS + 1):
        tu = f"{TARGET_STATION_PREFIX}_u_forecast_t_plus_{s}"
        tv = f"{TARGET_STATION_PREFIX}_v_forecast_t_plus_{s}"
        hu, hv = not preds[tu].isnull().all(), not preds[tv].isnull().all()
        mu = mean_absolute_error(y_test[tu], preds[tu]) if hu else np.nan
        su = mean_squared_error(y_test[tu], preds[tu]) if hu else np.nan
        mv = mean_absolute_error(y_test[tv], preds[tv]) if hv else np.nan
        sv = mean_squared_error(y_test[tv], preds[tv]) if hv else np.nan
        if hu: mae_s += mu; mse_s += su; cnt += 1
        if hv: mae_s += mv; mse_s += sv; cnt += 1
        rows.append({'Lead (t+)': s, 'MAE_u': mu, 'MSE_u': su, 'MAE_v': mv, 'MSE_v': sv,
                     'Avg_MAE': np.nanmean([mu, mv]), 'Avg_MSE': np.nanmean([su, sv])})

    metrics = pd.DataFrame(rows)
    print(f"\nOverall MAE: {mae_s/cnt:.4f}, MSE: {mse_s/cnt:.4f}")
    display(metrics.round(4))

    hours = metrics['Lead (t+)'] * 0.5
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 8))
    for ax, mu_col, mv_col, avg_col, yl in [(ax1, 'MAE_u', 'MAE_v', 'Avg_MAE', 'MAE'),
                                              (ax2, 'MSE_u', 'MSE_v', 'Avg_MSE', 'MSE')]:
        ax.plot(hours, metrics[mu_col], 'r--o', ms=4, label=f'{yl} U')
        ax.plot(hours, metrics[mv_col], 'b--s', ms=4, label=f'{yl} V')
        ax.plot(hours, metrics[avg_col], 'k-x', ms=5, lw=2.5, label=f'Avg {yl}')
        ax.set_xlabel('Lead Time (hours)'); ax.set_ylabel(yl)
        ax.set_xticks(np.arange(0, NUM_FORECAST_STEPS * 0.5 + 0.5, 2))
        ax.set_ylim(bottom=0); ax.grid(True, linestyle=':', alpha=0.7)
        ax.set_title(f'{yl} vs Lead Time'); ax.legend()
    fig.suptitle('Model Performance vs Forecast Lead Time', fontsize=18, y=0.98)
    plt.tight_layout(rect=[0, 0.02, 1, 0.94])

    if MODEL_EVAL_OUTPUT_DIR:
        ed = Path(MODEL_EVAL_OUTPUT_DIR); ed.mkdir(parents=True, exist_ok=True)
        plt.savefig(ed / "performance_vs_lead_time.png")
        metrics.to_csv(ed / "per_step_metrics.csv", index=False)
        print(f"Saved to {ed}")
    plt.show(); plt.close(fig)
else:
    print("RUN_EVALUATE is False. Skipping.")

## 2d: Plot Forecast Samples

Generates forecast vs actual comparison plots showing temperature, dewpoint, wind speed, wind direction.

In [ ]:
def uv_to_ws_wd(u, v):
    u, v = np.asarray(u, float), np.asarray(v, float)
    ws = np.sqrt(u**2 + (-v)**2)
    wd = (270 - np.degrees(np.arctan2(-v, u))) % 360
    wd[ws < 0.1] = 0
    return ws, wd

def get_preds(loaded, X, num_steps, prefix):
    pu, pv = np.full(num_steps, np.nan), np.full(num_steps, np.nan)
    for s in range(1, num_steps + 1):
        tu = f"{prefix}_u_forecast_t_plus_{s}"
        tv = f"{prefix}_v_forecast_t_plus_{s}"
        if tu in loaded: pu[s-1] = loaded[tu].predict(X)[0]
        if tv in loaded: pv[s-1] = loaded[tv].predict(X)[0]
    return pu, pv

def plot_forecast(anchor_ts, loaded, feat_row, yssy_df, x_cols,
                  plots_dir, plot_idx=""):
    t0, t1 = anchor_ts - timedelta(hours=24), anchor_ts + timedelta(hours=24)
    obs = yssy_df[(yssy_df[YSSY_OBS_TIMESTAMP_COL] >= t0) &
                  (yssy_df[YSSY_OBS_TIMESTAMP_COL] <= t1)].copy()
    if obs.empty:
        print(f"  No obs for {anchor_ts}"); return

    ot = obs[YSSY_OBS_TIMESTAMP_COL].values
    otemp = obs.get('air_temp', pd.Series(np.nan, index=obs.index)).values
    odewp = obs.get('dew_point', pd.Series(np.nan, index=obs.index)).values
    ou = obs.get('u_component', pd.Series(np.nan, index=obs.index)).values
    ov = obs.get('v_component', pd.Series(np.nan, index=obs.index)).values
    ows, owd = uv_to_ws_wd(ou, ov)

    ft = [anchor_ts + timedelta(minutes=30 * i) for i in range(1, NUM_FORECAST_STEPS + 1)]
    X_s = pd.DataFrame([feat_row[x_cols]], columns=x_cols)
    pu, pv = get_preds(loaded, X_s, NUM_FORECAST_STEPS, TARGET_STATION_PREFIX)
    pws, pwd = uv_to_ws_wd(pu, pv)

    fig, ax1 = plt.subplots(figsize=(18, 10))
    fig.suptitle(f"YSSY Forecast ({anchor_ts.strftime('%Y-%m-%d %H:%M')})", fontsize=16)
    ax1.plot(ot, otemp, 'r-', label='Obs Temp', zorder=3)
    ax1.plot(ot, odewp, 'b-', label='Obs Dewpoint', zorder=3)
    ax1.plot(ot, ows, '-', color='gray', lw=3, label='Obs Wind Speed', zorder=3)
    ax1.plot(ft, pws, '-', color='black', lw=4, label='Fcst Wind Speed', zorder=4)
    ax1.set_ylabel('Temp (C) / Speed (kt)'); ax1.grid(True, ls=':', alpha=0.7)
    ax1.yaxis.set_major_locator(mticker.MultipleLocator(5)); ax1.set_ylim(0, 40)

    ax2 = ax1.twinx()
    ax2.scatter(ot, owd, marker='o', color='darkgray', s=50, label='Obs Wind Dir', zorder=6)
    ax2.scatter(ft, pwd, marker='X', color='black', s=70, label='Fcst Wind Dir', zorder=7)
    ax2.set_ylabel('Direction (deg)'); ax2.set_ylim(0, 360)
    ax2.yaxis.set_major_locator(mticker.MultipleLocator(45))

    ax1.set_xlim(t0, t1)
    ax1.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M\n%d-%b'))
    ax1.xaxis.set_major_locator(mdates.HourLocator(interval=2))
    ax1.axvline(anchor_ts, color='k', ls=':', lw=1, alpha=0.7, label='Anchor')

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax2.legend(h1 + h2, l1 + l2, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3, frameon=False)
    plt.tight_layout(rect=[0, 0.05, 1, 0.95])

    if plots_dir:
        pd_ = Path(plots_dir); pd_.mkdir(parents=True, exist_ok=True)
        f = pd_ / f"forecast_{anchor_ts.strftime('%Y%m%d_%H%M')}{plot_idx}.png"
        plt.savefig(f); print(f"  Saved: {f}")
    plt.show(); plt.close(fig)


if RUN_PLOT:
    assert MODEL_OUTPUT_DIR and MODEL_TEST_FILE and YSSY_OBS_FILE and MODEL_PLOTS_OUTPUT_DIR
    loaded, feat_names = load_all_models(MODEL_OUTPUT_DIR)
    if not loaded:
        raise RuntimeError("No models loaded.")

    test_df = pd.read_csv(MODEL_TEST_FILE)
    test_df[MODEL_TIMESTAMP_COL] = pd.to_datetime(test_df[MODEL_TIMESTAMP_COL])
    print(f"Test: {test_df.shape}")

    yssy_df = pd.read_csv(YSSY_OBS_FILE)
    yssy_df[YSSY_OBS_TIMESTAMP_COL] = pd.to_datetime(yssy_df[YSSY_OBS_TIMESTAMP_COL])
    yssy_df.set_index(YSSY_OBS_TIMESTAMP_COL, inplace=True, drop=False)
    yssy_df.sort_index(inplace=True)
    print(f"YSSY obs: {yssy_df.shape}")

    x_cols = feat_names if feat_names else \
        [c for c in test_df.columns if c != MODEL_TIMESTAMP_COL
         and not c.startswith(f"{TARGET_STATION_PREFIX}_")]

    n = min(NUM_RANDOM_PLOT_SAMPLES, len(test_df))
    for i, idx in enumerate(random.sample(range(len(test_df)), n)):
        row = test_df.iloc[idx]
        print(f"  Sample {i+1}/{n}: {row[MODEL_TIMESTAMP_COL]}")
        plot_forecast(row[MODEL_TIMESTAMP_COL], loaded, row, yssy_df,
                      x_cols, MODEL_PLOTS_OUTPUT_DIR, f"_rand{i+1}")

    if PLOT_SPECIFIC_DATE_RANGE:
        s_dt = pd.to_datetime(PLOT_START_DATETIME)
        e_dt = pd.to_datetime(PLOT_END_DATETIME)
        for t in pd.date_range(s_dt, e_dt, freq='30min'):
            match = test_df[test_df[MODEL_TIMESTAMP_COL] == t]
            if not match.empty:
                plot_forecast(t, loaded, match.iloc[0], yssy_df,
                              x_cols, MODEL_PLOTS_OUTPUT_DIR,
                              f"_auto_{t.strftime('%Y%m%d_%H%M')}")
else:
    print("RUN_PLOT is False. Skipping.")